# 02 SQL with DuckDB
Five queries, then three traps.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/GITHUB_OWNER/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==2.0.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("KIT_DATA_DIR", ROOT / "data"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-02"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder:", ROOT)

Step 2: load users and events

In [ ]:
import duckdb
con = duckdb.connect()
con.execute(f"CREATE TABLE users AS SELECT * FROM read_csv('{DATA / 'users.csv'}')")
con.execute(f"CREATE TABLE events AS SELECT * FROM read_csv('{DATA / 'events.csv'}')")
q = lambda sql: con.sql(sql).df()        # run a query, get a table
one = lambda sql: con.execute(sql).fetchone()[0]  # run a query, get one number
q("SELECT * FROM users LIMIT 5")

Step 3: query 1, filter

In [ ]:
q("SELECT * FROM events WHERE event_name = 'export' LIMIT 5")

Step 4: query 2, group by

In [ ]:
q("SELECT event_name, COUNT(*) AS events FROM events GROUP BY event_name ORDER BY events DESC")

Step 5: query 3, join (one row per user first)

In [ ]:
con.execute("""CREATE VIEW current_plan AS
SELECT user_id, plan FROM users
QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY plan_start DESC) = 1""")
q("""SELECT COALESCE(c.plan, 'unknown') AS plan, COUNT(*) AS events
FROM events e JOIN current_plan c USING (user_id) GROUP BY 1 ORDER BY 2 DESC""")

Step 6: query 4, date range (October 2026)

In [ ]:
q("""SELECT COUNT(*) AS october_events FROM events
WHERE event_ts >= TIMESTAMP '2026-10-01' AND event_ts < TIMESTAMP '2026-11-01'""")

Step 7: query 5, a rate with an honest denominator

In [ ]:
q("""SELECT
  COUNT(DISTINCT e.user_id) AS users_who_clicked_upgrade,
  (SELECT COUNT(DISTINCT user_id) FROM users) AS all_users,
  ROUND(COUNT(DISTINCT e.user_id) / (SELECT COUNT(DISTINCT user_id) FROM users), 3) AS rate
FROM events e
WHERE e.event_name = 'upgrade_click'
  AND e.event_ts >= TIMESTAMP '2026-10-01' AND e.event_ts < TIMESTAMP '2026-11-01'""")

Step 8: trap 1, a join that fans out

In [ ]:
wrong = one("SELECT COUNT(*) FROM events e JOIN users u USING (user_id)")
right = one("SELECT COUNT(*) FROM events e JOIN current_plan c USING (user_id)")
print("Wrong (join to every plan row):", wrong, "| Right (one row per user):", right)
assert wrong != right, "The two numbers should differ"

Step 9: trap 2, NULL plans dropped by WHERE

In [ ]:
total = one("SELECT COUNT(*) FROM current_plan")
free = one("SELECT COUNT(*) FROM current_plan WHERE plan = 'free'")
wrong = one("SELECT COUNT(*) FROM current_plan WHERE plan <> 'free'")
right = one("SELECT COUNT(*) FROM current_plan WHERE plan IS DISTINCT FROM 'free'")
print(f"Users: {total} | free: {free} | not free (wrong): {wrong} | not free (right): {right}")
print("Wrong free + not free =", free + wrong, "which is less than", total)
assert wrong != right, "The two numbers should differ"

Step 10: trap 3, the midnight boundary

In [ ]:
wrong_a = one("SELECT COUNT(*) FROM events WHERE event_ts BETWEEN '2026-10-01' AND '2026-10-31'")
wrong_b = one("SELECT COUNT(*) FROM events WHERE event_ts BETWEEN '2026-10-01' AND '2026-11-01'")
right = one("SELECT COUNT(*) FROM events WHERE event_ts >= '2026-10-01' AND event_ts < '2026-11-01'")
print("BETWEEN Oct 1 and Oct 31 (misses most of Oct 31):", wrong_a)
print("BETWEEN Oct 1 and Nov 1 (counts Nov 1 00:00:00):", wrong_b)
print("Half open, >= Oct 1 and < Nov 1 (right):", right)
assert wrong_a != right and wrong_b != right, "The numbers should differ"